# 🔍 Projeto Íris - Web Scraper Abrangente de Vagas de Emprego

Este notebook realiza scraping de **TODOS** os principais sites de vagas de emprego no Brasil, incluindo:

### Sites Generalistas:
- **Catho** - Um dos maiores portais de emprego
- **InfoJobs** - Portal consolidado de vagas
- **Vagas.com** - Grande portal de empregos
- **Trabalha Brasil** - Portal do governo
- **Indeed** - Agregador global de vagas
- **LinkedIn** - Rede profissional
- **Glassdoor** - Vagas e avaliações de empresas
- **Jooble** - Agregador de vagas
- **Empregos.com.br** - Portal de empregos

### Sites de Tech/Freelance:
- **Programathor** - Vagas de tecnologia
- **APInfo** - Vagas de TI
- **GeekHunter** - Vagas tech
- **99Freelas** - Freelance e projetos
- **Workana** - Freelance internacional
- **GetNinjas** - Serviços e freelance

### Sites de Classificados:
- **OLX** - Classificados gerais
- **Quinto Andar** - Vagas imobiliárias
- **Mercado Livre** - Vagas em comércio

### Sites Especializados:
- **RemoteOK** - Vagas remotas internacionais
- **Trampos.co** - Vagas criativas
- **Curriculum.com.br** - Portal de empregos

---

### Autores
João Alex (SSIT) e Pedro Henrique (IEEE CIS)

## 📦 Instalação de Dependências

In [1]:
# Instalação das bibliotecas necessárias
!pip install requests beautifulsoup4 pandas lxml selenium fake-useragent cloudscraper


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 🛠️ Imports e Configuração

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import quote, urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import time
import random
from datetime import datetime
import re
from fake_useragent import UserAgent
import warnings
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
warnings.filterwarnings('ignore')

# Configuração de Logs
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger("IrisScraper")

# User Agent dinâmico
ua = UserAgent()

def get_headers():
    """Gera headers dinâmicos para evitar bloqueios"""
    return {
        'User-Agent': ua.random,
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
        'Accept-Encoding': 'gzip, deflate, br',
        'DNT': '1',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'
    }

def safe_request(url, max_retries=3):
    """Faz requisição com retry automático"""
    for attempt in range(max_retries):
        try:
            time.sleep(random.uniform(1, 3))  # Rate limiting
            response = requests.get(url, headers=get_headers(), timeout=15, verify=False)
            response.raise_for_status()
            return response
        except Exception as e:
            if attempt == max_retries - 1:
                logger.warning(f"Falha após {max_retries} tentativas: {url}")
                return None
            time.sleep(random.uniform(2, 5))
    return None

## 🌐 Scrapers por Site

### 1. CATHO

In [ ]:
def scrape_catho(keyword):
    """Scraper para Catho.com.br"""
    source_name = "Catho"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        # URL de busca
        keyword_encoded = quote(keyword)
        url = f"https://www.catho.com.br/vagas/{keyword_encoded}/"

        driver = webdriver.Firefox()
        driver.get(url)
        
        page = 1
        max_pages = -1 # will fetch this as soon as it reaches the end of the page

        if page == 1:
            #Aceitar Cookies
            WebDriverWait(driver, 60).until(EC.element_to_be_clickable((By.XPATH, "//*[text() = 'Aceitar todos os cookies' ]"))).click()
        else:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "article")))

        while page != max_pages + 1:

            job_cards = driver.find_elements(By.TAG_NAME, "article")
            for card in job_cards:
                try:
                    #Título da Vaga
                    title_elem = card.find_element(By.TAG_NAME, "h2") or card.find_element(By.TAG_NAME, "h3")
                    if not title_elem:
                        continue

                    title = title_elem.text

                    #Link da Vaga
                    link_elem = card.find_element(By.TAG_NAME, 'a')
                    href = link_elem.get_attribute('href')

                    link = urljoin('https://www.catho.com.br', href) if link_elem else url
                    
                    #Focar Elemento
                    driver.execute_script("arguments[0].scrollIntoView();", card)            
                    card.click()
                    
                    # Empresa
                    company_elem = driver.find_element(By.CLASS_NAME, 'text-neutral')
                    company = company_elem.text if company_elem else "Não informado"    
                    
                    salary_job_quantity = card.find_elements(By.TAG_NAME, 'strong')
                    salary = ""
                    quantity = ""
                    # Qtd Vagas e Salário
                    if "R$" in salary_job_quantity[0].text:
                        quantity = salary_job_quantity[1].text
                        salary = salary_job_quantity[0].text
                    else:
                        quantity = salary_job_quantity[0].text
                        salary = salary_job_quantity[1].text                 

                    # Localização
                    location_elem = card.find_element(By.TAG_NAME, 'strong').find_element(By.XPATH, "..")
                    location = location_elem.text if location_elem else "Brasil"

                    #Descricao
                    description_elem = driver.find_element(By.CLASS_NAME, "whitespace-pre-line")
                    descrpition = description_elem.text if description_elem else "Não informado"
                    
                    vagas.append({
                        'Fonte': source_name,
                        'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                        'Cargo': title,
                        'Qtd Vagas': quantity,
                        'Descrição': descrpition,
                        'Empresa': company,
                        'Local': location,
                        'Salário': salary,
                        'Link': link
                    })
                except Exception as e:
                    continue

            if max_pages == -1:
                max_page_elem = driver.find_element(By.XPATH, "//*[@class='page-button buildLink ']") 
                print(max_page_elem.text)
                max_pages = int(max_page_elem.text)
                if page == max_pages: 
                    break
            page += 1
            next_page_elem = driver.find_element(By.CLASS_NAME, "next-page")
            next_page_elem.click()

    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas
    

20:55:51 - 🕷️ [Catho] Iniciando busca...


4


20:56:39 - ✅ [Catho] 58 vagas encontradas.


[{'Fonte': 'Catho',
  'Data_Coleta': '2026-07-20 20:56',
  'Cargo': 'Berçarista',
  'Qtd Vagas': '1 vaga',
  'Descrição': 'Irá atuar com crianças com idades entre 1 e2 anos, auxiliando-as na higiene, alimentação e na organização dos ambientes e pertences.Necessário Ensino Médio Completo e possuir o Histórico Escolar.\nNecessário experiência na função e disponibilidade para realizar horas extras quando necessário.\nImportante residir na Zona Sul de SP, e ter acesso fácil à região do Aeroporto de Congonhas ou Shopping Interlagos.',
  'Empresa': 'BABY NURSE',
  'Local': '1 vaga - São Paulo',
  'Salário': 'R$ 2.030',
  'Link': 'https://www.catho.com.br/vagas/bercarista/37587412'},
 {'Fonte': 'Catho',
  'Data_Coleta': '2026-07-20 20:56',
  'Cargo': 'Babá',
  'Qtd Vagas': '1 vaga',
  'Descrição': 'Cuidar de uma criança idade 8 anos, com atividades recreativas\nadequadas( Jogar bola, Jogos de tabuleiro, etc.);\nPreparar e servir lanches/refeições, lavar e passar (tudo que se\nrefere a criança

### 2. INFOJOBS

In [ ]:
def scrape_infojobs(keyword):
    """Scraper para InfoJobs.com.br"""
    source_name = "InfoJobs"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword.replace(' ', '+'))
        url = f"https://www.infojobs.com.br/vagas-de-emprego-{keyword_encoded}.aspx"
        
        driver =  webdriver.Firefox()
        driver.get(url)
        
        #Aceitar cookies e remover Pop-Up
        WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, "//*[ text() = 'Aceitar' ]"))).click()
        WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, "//*[ contains(text(),'Agora não')]"))).click()
        
        
        #Carregar toda a página antes
        last_height = driver.execute_script("return document.documentElement.scrollHeight")
        while True:
            driver.execute_script("window.scrollTo(0,document.documentElement.scrollHeight);")
            time.sleep(3)
            new_height = driver.execute_script("return document.documentElement.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        # Buscar vagas
        job_elements = driver.find_elements(By.XPATH, "//*[@class='pt-24 px-24 cursor-pointer js_vacancyLoad js_rowCard js_cardLink']") 
        
        for job in job_elements:
            try:
                #Focar Elemento
                driver.execute_script("arguments[0].scrollIntoView();", job)            
                job.click()

                title_elem = job.find_element(By.TAG_NAME, 'h2')
                if not title_elem:
                    continue
                    
                title = title_elem.text

                #Link da Vaga
                link_elem = job.find_element(By.TAG_NAME, 'a')
                href = link_elem.get_attribute('href')

                link = urljoin('https://www.infojobs.com.br', href) if link_elem else url      

                
                # Localização e data
                location_elem = job.find_element(By.CLASS_NAME, 'mb-8')
                location = location_elem.gtext if location_elem else "Brasil"

                company_elem = job.find_element(By.CLASS_NAME, "text-body")
                company = company_elem.text
                
                vagas.append({
                    'Fonte': source_name,
                    'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                    'Cargo': title,
                    'Empresa': "Ver no link",
                    'Local': location,
                    'Link': link
                })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

scrape_infojobs("cuidadora")

17:07:12 - 🕷️ [InfoJobs] Iniciando busca...


### 3. VAGAS.COM

In [ ]:
def scrape_vagas_com(keyword):
    """Scraper para Vagas.com"""
    source_name = "Vagas.com"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword.replace(' ', '-'))
        url = f"https://www.vagas.com.br/vagas-de-{keyword_encoded}"
        
        response = safe_request(url)
        if not response:
            return vagas
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Buscar vagas
        job_list = soup.select('li.vaga') or soup.find_all('li', class_=re.compile('job|vaga'))
        
        for job in job_list[:30]:
            try:
                # Título
                title_elem = job.find('a', class_=re.compile('link-detalhes-vaga'))
                if not title_elem:
                    continue
                    
                title = title_elem.get('title', title_elem.get_text(strip=True))
                link = f"https://www.vagas.com.br{title_elem['href']}" if title_elem.get('href') else url
                
                # Local
                location_elem = job.find('span', class_=re.compile('vaga-local'))
                location = location_elem.get_text(strip=True) if location_elem else "Brasil"
                
                vagas.append({
                    'Fonte': source_name,
                    'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                    'Cargo': title,
                    'Empresa': "Confidencial",
                    'Local': location,
                    'Link': link
                })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

20:38:29 - 🕷️ [Vagas.com] Iniciando busca...


KeyboardInterrupt: 

### 4. TRABALHA BRASIL

In [ ]:
def scrape_trabalha_brasil(keyword):
    """Scraper para Trabalha Brasil (Portal do Governo)"""
    source_name = "Trabalha Brasil"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword.replace(' ', '-'))
        url = f"https://www.trabalhabrasil.com.br/vagas-empregos/{keyword_encoded}"
        
        response = safe_request(url)
        if not response:
            return vagas
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Buscar vagas
        cards = soup.select('div.job__container') or soup.find_all('div', class_=re.compile('job'))
        
        for card in cards[:30]:
            try:
                title_elem = card.find('h2', class_='job__name')
                if not title_elem:
                    continue
                    
                title = title_elem.get_text(strip=True)
                
                link_elem = card.find('a', href=True)
                link = urljoin('https://www.trabalhabrasil.com.br', link_elem['href']) if link_elem else url
                
                company_elem = card.find('h3', class_='job__company')
                company = company_elem.get_text(strip=True) if company_elem else "Confidencial"
                
                vagas.append({
                    'Fonte': source_name,
                    'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                    'Cargo': title,
                    'Empresa': company,
                    'Local': "Verificar no Link",
                    'Link': link
                })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 5. INDEED

In [ ]:
def scrape_indeed(keyword):
    """Scraper para Indeed.com.br"""
    source_name = "Indeed"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword)
        url = f"https://br.indeed.com/jobs?q={keyword_encoded}&l=Brasil"
        
        response = safe_request(url)
        if not response:
            return vagas
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Buscar vagas
        job_cards = soup.select('div.job_seen_beacon') or \
                   soup.select('div.jobsearch-SerpJobCard') or \
                   soup.find_all('div', class_=re.compile('job'))
        
        for card in job_cards[:30]:
            try:
                # Título
                title_elem = card.find('h2', class_=re.compile('jobTitle')) or card.find('a', class_=re.compile('jcs-JobTitle'))
                if not title_elem:
                    continue
                    
                title = title_elem.get_text(strip=True)
                
                # Link
                link_elem = title_elem.find('a') if title_elem.name != 'a' else title_elem
                link = urljoin('https://br.indeed.com', link_elem['href']) if link_elem and link_elem.get('href') else url
                
                # Empresa
                company_elem = card.find('span', class_=re.compile('companyName'))
                company = company_elem.get_text(strip=True) if company_elem else "Não informado"
                
                # Local
                location_elem = card.find('div', class_=re.compile('companyLocation'))
                location = location_elem.get_text(strip=True) if location_elem else "Brasil"
                
                vagas.append({
                    'Fonte': source_name,
                    'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                    'Cargo': title,
                    'Empresa': company,
                    'Local': location,
                    'Link': link
                })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 6. LINKEDIN

In [ ]:
def scrape_linkedin(keyword):
    """Scraper para LinkedIn (limitado sem autenticação)"""
    source_name = "LinkedIn"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword)
        url = f"https://www.linkedin.com/jobs/search?keywords={keyword_encoded}&location=Brasil"
        
        response = safe_request(url)
        if not response:
            return vagas
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # LinkedIn é mais difícil sem autenticação, mas ainda podemos pegar alguns dados
        job_cards = soup.select('div.base-card') or soup.find_all('li', class_=re.compile('job'))
        
        for card in job_cards[:20]:
            try:
                title_elem = card.find('h3', class_=re.compile('base-search-card__title'))
                if not title_elem:
                    continue
                    
                title = title_elem.get_text(strip=True)
                
                link_elem = card.find('a', class_=re.compile('base-card__full-link'))
                link = link_elem['href'] if link_elem and link_elem.get('href') else url
                
                company_elem = card.find('h4', class_=re.compile('base-search-card__subtitle'))
                company = company_elem.get_text(strip=True) if company_elem else "Não informado"
                
                location_elem = card.find('span', class_=re.compile('job-search-card__location'))
                location = location_elem.get_text(strip=True) if location_elem else "Brasil"
                
                vagas.append({
                    'Fonte': source_name,
                    'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                    'Cargo': title,
                    'Empresa': company,
                    'Local': location,
                    'Link': link
                })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 7. PROGRAMATHOR (Tech)

In [ ]:
def scrape_programathor(keyword):
    """Scraper para Programathor (vagas de tecnologia)"""
    source_name = "Programathor"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword.lower().replace(' ', '-'))
        url = f"https://programathor.com.br/jobs/{keyword_encoded}"
        
        response = safe_request(url)
        if not response:
            return vagas
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        job_cards = soup.select('div.cell-list')
        
        for card in job_cards[:30]:
            try:
                title = card.find('h3').get_text(strip=True)
                link = urljoin('https://programathor.com.br', card.find('a')['href'])
                
                location_elem = card.find('span', class_='cell-list-content-icon-location')
                location = location_elem.get_text(strip=True) if location_elem else "Remoto/Brasil"
                
                vagas.append({
                    'Fonte': source_name,
                    'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                    'Cargo': title,
                    'Empresa': "Confidencial",
                    'Local': location,
                    'Link': link
                })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 8. OLX

In [ ]:
def scrape_olx(keyword):
    """Scraper para OLX (classificados)"""
    source_name = "OLX"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword)
        url = f"https://www.olx.com.br/vagas-de-emprego?q={keyword_encoded}"
        
        response = safe_request(url)
        if not response:
            return vagas
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # OLX usa estrutura dinâmica, tentar múltiplos seletores
        items = soup.find_all('li', class_=lambda x: x and 'sc-' in str(x)) or \
               soup.select('div[data-ds-component="DS-AdCard"]')
        
        for item in items[:30]:
            try:
                link_tag = item.find('a', href=True)
                if not link_tag:
                    continue
                    
                title_tag = link_tag.find('h2') or link_tag.find('span')
                if title_tag:
                    title = title_tag.get_text(strip=True)
                    link = link_tag['href']
                    if not link.startswith('http'):
                        link = f"https://www.olx.com.br{link}"
                    
                    vagas.append({
                        'Fonte': source_name,
                        'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                        'Cargo': title,
                        'Empresa': "Anunciante OLX",
                        'Local': "Verificar Link",
                        'Link': link
                    })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 9. REMOTEOK (Vagas Remotas - API)

In [ ]:
def scrape_remoteok(keyword):
    """Scraper para RemoteOK (API pública)"""
    source_name = "RemoteOK"
    logger.info(f"🕷️ [{source_name}] Iniciando busca via API...")
    vagas = []
    
    try:
        response = safe_request("https://remoteok.com/api")
        if not response:
            return vagas
            
        data = response.json()
        
        # O primeiro item é informação legal, pulamos
        for item in data[1:]:
            try:
                position = item.get('position', '').lower()
                if keyword.lower() in position or keyword == "":
                    vagas.append({
                        'Fonte': source_name,
                        'Data_Coleta': item.get('date', '')[:10],
                        'Cargo': item.get('position'),
                        'Empresa': item.get('company'),
                        'Local': item.get('location', 'Remoto'),
                        'Link': item.get('url')
                    })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 10. GUPY (API)

In [ ]:
def scrape_gupy(keyword):
    """Scraper para Gupy (portal usado por muitas empresas)"""
    source_name = "Gupy"
    logger.info(f"🕷️ [{source_name}] Iniciando busca via API...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword)
        url = f"https://employability-portal.gupy.io/api/v1/jobs?jobName={keyword_encoded}&offset=0&limit=50"
        
        headers = get_headers()
        headers.update({
            'Accept': 'application/json',
            'Origin': 'https://portal.gupy.io',
            'Referer': 'https://portal.gupy.io/'
        })
        
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            data = response.json()
            jobs = data.get('data', [])
            
            for job in jobs:
                try:
                    vagas.append({
                        'Fonte': source_name,
                        'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                        'Cargo': job.get('name', 'N/A'),
                        'Empresa': job.get('careerPageName', 'Não informado'),
                        'Local': f"{job.get('city', '')}, {job.get('state', '')}".strip(', '),
                        'Link': f"https://portal.gupy.io/job/{job.get('id')}"
                    })
                except:
                    continue
                    
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 11. JOOBLE

In [ ]:
def scrape_jooble(keyword):
    """Scraper para Jooble (agregador de vagas)"""
    source_name = "Jooble"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword)
        url = f"https://br.jooble.org/desc-{keyword_encoded}/trabalhos"
        
        response = safe_request(url)
        if not response:
            return vagas
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        job_cards = soup.select('article.vacancy-wrapper') or \
                   soup.find_all('div', class_=re.compile('job|vacancy'))
        
        for card in job_cards[:30]:
            try:
                title_elem = card.find('a', class_=re.compile('position-card-title'))
                if not title_elem:
                    continue
                    
                title = title_elem.get_text(strip=True)
                link = urljoin('https://br.jooble.org', title_elem['href']) if title_elem.get('href') else url
                
                company_elem = card.find('span', class_=re.compile('company-name'))
                company = company_elem.get_text(strip=True) if company_elem else "Não informado"
                
                location_elem = card.find('div', class_=re.compile('location'))
                location = location_elem.get_text(strip=True) if location_elem else "Brasil"
                
                vagas.append({
                    'Fonte': source_name,
                    'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                    'Cargo': title,
                    'Empresa': company,
                    'Local': location,
                    'Link': link
                })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

### 12. TRAMPOS.CO

In [ ]:
def scrape_trampos(keyword):
    """Scraper para Trampos.co (vagas criativas)"""
    source_name = "Trampos.co"
    logger.info(f"🕷️ [{source_name}] Iniciando busca...")
    vagas = []
    
    try:
        keyword_encoded = quote(keyword)
        url = f"https://www.trampos.co/oportunidades?q={keyword_encoded}"
        
        response = safe_request(url)
        if not response:
            return vagas
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        job_cards = soup.select('li.opportunity') or \
                   soup.find_all('div', class_=re.compile('opportunity|job'))
        
        for card in job_cards[:30]:
            try:
                title_elem = card.find('h3') or card.find('h4')
                if not title_elem:
                    continue
                    
                title = title_elem.get_text(strip=True)
                
                link_elem = card.find('a', href=True)
                link = urljoin('https://www.trampos.co', link_elem['href']) if link_elem else url
                
                company_elem = card.find('span', class_=re.compile('company'))
                company = company_elem.get_text(strip=True) if company_elem else "Não informado"
                
                vagas.append({
                    'Fonte': source_name,
                    'Data_Coleta': datetime.now().strftime('%Y-%m-%d %H:%M'),
                    'Cargo': title,
                    'Empresa': company,
                    'Local': "Brasil",
                    'Link': link
                })
            except:
                continue
                
    except Exception as e:
        logger.error(f"❌ [{source_name}] Erro: {e}")
        
    logger.info(f"✅ [{source_name}] {len(vagas)} vagas encontradas.")
    return vagas

## 🚀 Orquestrador Principal

In [16]:
def executar_pipeline_completo(keyword, max_workers=8):
    """
    Executa TODOS os scrapers em paralelo
    
    Args:
        keyword: Termo de busca (ex: 'cientista de dados', 'babá', 'modelo')
        max_workers: Número de scrapers simultâneos
    
    Returns:
        DataFrame com todas as vagas encontradas
    """
    logger.info("="*80)
    logger.info(f"🚀 INICIANDO PIPELINE COMPLETO - Busca: '{keyword}'")
    logger.info("="*80)
    
    # Lista de todos os scrapers
    scrapers = [
        scrape_catho,
        #scrape_infojobs,
        #scrape_vagas_com,
        #scrape_trabalha_brasil,
        #scrape_indeed,
        #scrape_linkedin,
        #scrape_programathor,
        #scrape_olx,
        #scrape_remoteok,
        #scrape_gupy,
        #scrape_jooble,
        #scrape_trampos
    ]
    
    todos_resultados = []
    
    # Execução paralela
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Agendar todas as buscas
        futures = {executor.submit(scraper, keyword): scraper.__name__ for scraper in scrapers}
        
        # Processar resultados conforme completam
        for future in as_completed(futures):
            scraper_name = futures[future]
            try:
                resultado = future.result()
                if resultado:
                    todos_resultados.extend(resultado)
                    logger.info(f"✓ {scraper_name} completado")
            except Exception as e:
                logger.error(f"✗ Erro em {scraper_name}: {e}")
    
    # Criar DataFrame
    if not todos_resultados:
        logger.error("❌ Nenhuma vaga encontrada em nenhum site.")
        return pd.DataFrame()
    
    df = pd.DataFrame(todos_resultados)
    
    # Remover duplicatas baseadas no Link
    df.drop_duplicates(subset=['Link'], inplace=True)
    
    # Estatísticas por fonte
    logger.info("="*80)
    logger.info("📊 ESTATÍSTICAS POR FONTE:")
    stats = df['Fonte'].value_counts()
    for fonte, count in stats.items():
        logger.info(f"   {fonte}: {count} vagas")
    
    logger.info("="*80)
    logger.info(f"🏁 TOTAL CONSOLIDADO: {len(df)} vagas únicas encontradas")
    logger.info("="*80)
    
    return df

## 🎯 EXECUÇÃO

### Configure sua busca aqui:

In [ ]:
# ⚠️ CONFIGURE AQUI O TERMO DE BUSCA
TERMO_BUSCA = "cientista de dados"  # Mude para: 'babá', 'modelo', 'recepcionista', etc.

# Executar o pipeline completo
df_vagas = executar_pipeline_completo(TERMO_BUSCA, max_workers=6)

# Verificar se encontrou vagas
if not df_vagas.empty:
    print(f"\n{'='*80}")
    print(f"✅ SUCESSO! {len(df_vagas)} vagas encontradas")
    print(f"{'='*80}\n")
else:
    print("\n⚠️ Nenhuma vaga encontrada. Tente outro termo de busca.\n")

## 📊 Visualização dos Dados

In [ ]:
if not df_vagas.empty:
    # Configurar display do pandas
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 100)
    pd.set_option('display.width', None)
    
    # Mostrar primeiras vagas
    print("\n📋 PRIMEIRAS 20 VAGAS ENCONTRADAS:")
    print("="*120)
    display(df_vagas.head(20))
    
    # Estatísticas gerais
    print("\n📈 ESTATÍSTICAS GERAIS:")
    print("="*80)
    print(f"Total de vagas: {len(df_vagas)}")
    print(f"Número de fontes: {df_vagas['Fonte'].nunique()}")
    print(f"Empresas únicas: {df_vagas['Empresa'].nunique()}")
    print(f"\nDistribuição por fonte:")
    print(df_vagas['Fonte'].value_counts())
    
    # Top empresas
    print(f"\n🏢 TOP 10 EMPRESAS COM MAIS VAGAS:")
    print("="*80)
    top_empresas = df_vagas[df_vagas['Empresa'] != 'Confidencial']['Empresa'].value_counts().head(10)
    for empresa, count in top_empresas.items():
        print(f"{empresa}: {count} vagas")

## 💾 Salvar Resultados

In [ ]:
if not df_vagas.empty:
    # Gerar nome do arquivo com timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'vagas_completo_{TERMO_BUSCA.replace(" ", "_")}_{timestamp}.csv'
    
    # Salvar CSV
    df_vagas.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n💾 Arquivo salvo: {filename}")
    print(f"   Total de registros: {len(df_vagas)}")
    print(f"   Colunas: {', '.join(df_vagas.columns)}")
    
    # Também salvar em Excel (opcional)
    try:
        excel_filename = filename.replace('.csv', '.xlsx')
        df_vagas.to_excel(excel_filename, index=False, engine='openpyxl')
        print(f"\n📊 Também salvo em Excel: {excel_filename}")
    except:
        print("\n⚠️ Não foi possível salvar em Excel (instale openpyxl se necessário)")

## 🔍 Busca Avançada - Múltiplos Termos

Para o Projeto Íris, podemos buscar múltiplos termos de risco:

In [17]:
# BUSCA MÚLTIPLA - Para análise de risco do Projeto Íris
termos_risco = [
    'babá',
    'modelo',
    'recepcionista',
    'promotora',
    'garçonete',
    'atendente',
    'cuidadora',
    'secretária'
]

print("🔍 INICIANDO BUSCA MÚLTIPLA PARA ANÁLISE DE RISCO")
print(f"Termos a buscar: {', '.join(termos_risco)}")
print("="*80)

# Coletar todas as vagas
df_todos_termos = pd.DataFrame()

for termo in termos_risco:
    print(f"\n🔎 Buscando: {termo}")
    df_temp = executar_pipeline_completo(termo, max_workers=4)
    if not df_temp.empty:
        df_temp['Termo_Busca'] = termo
        df_todos_termos = pd.concat([df_todos_termos, df_temp], ignore_index=True)
    print(f"   Encontradas: {len(df_temp)} vagas")
    time.sleep(5)  # Pausa entre buscas

# Remover duplicatas globais
df_todos_termos.drop_duplicates(subset=['Link'], inplace=True)

# Salvar dataset consolidado
if not df_todos_termos.empty:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename_consolidado = f'dataset_iris_completo_{timestamp}.csv'
    df_todos_termos.to_csv(filename_consolidado, index=False, encoding='utf-8-sig')
    
    print("\n" + "="*80)
    print("✅ BUSCA MÚLTIPLA CONCLUÍDA!")
    print(f"Total de vagas coletadas: {len(df_todos_termos)}")
    print(f"Arquivo salvo: {filename_consolidado}")
    print("\nDistribuição por termo de busca:")
    print(df_todos_termos['Termo_Busca'].value_counts())
    print("="*80)

🔍 INICIANDO BUSCA MÚLTIPLA PARA ANÁLISE DE RISCO
Termos a buscar: babá, modelo, recepcionista, promotora, garçonete, atendente, cuidadora, secretária


20:59:41 - ================================================================================
20:59:41 - 🚀 INICIANDO PIPELINE COMPLETO - Busca: 'babá'
20:59:41 - ================================================================================
20:59:41 - 🕷️ [Catho] Iniciando busca...



🔎 Buscando: babá
4


21:00:33 - ✅ [Catho] 56 vagas encontradas.
21:00:34 - ✓ scrape_catho completado
21:00:44 - ================================================================================
21:00:44 - 📊 ESTATÍSTICAS POR FONTE:
21:00:44 -    Catho: 55 vagas
21:00:44 - ================================================================================
21:00:44 - 🏁 TOTAL CONSOLIDADO: 55 vagas únicas encontradas
21:00:44 - ================================================================================


   Encontradas: 55 vagas


...

   Encontradas: 200 vagas


21:20:10 - ================================================================================
21:20:10 - 🚀 INICIANDO PIPELINE COMPLETO - Busca: 'secretária'
21:20:10 - ================================================================================
21:20:10 - 🕷️ [Catho] Iniciando busca...



🔎 Buscando: secretária
27


21:24:35 - ❌ [Catho] Erro: Message: Element <a class="next-page" href="/vagas/secret%C3%A1ria/?page=24"> could not be scrolled into view; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementnotinteractableexception
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
ElementNotInteractableError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:371:5
webdriverClickElement@chrome://remote/content/marionette/interaction.sys.mjs:165:11
interaction.clickElement@chrome://remote/content/marionette/interaction.sys.mjs:134:11
clickElement@chrome://remote/content/marionette/actors/MarionetteCommandsChild.sys.mjs:328:29
receiveMessage@chrome://remote/content/marionette/actors/MarionetteCommandsChild.sys.mjs:210:31

21:24:36 - ✅ [Catho] 410 vagas encontradas.
21:24:36 - ✓ scrape_catho completado
21:24:38 - ===========

   Encontradas: 396 vagas

✅ BUSCA MÚLTIPLA CONCLUÍDA!
Total de vagas coletadas: 2676
Arquivo salvo: dataset_iris_completo_20260720_212443.csv

Distribuição por termo de busca:
Termo_Busca
garçonete        925
secretária       394
atendente        358
promotora        313
recepcionista    218
modelo           215
cuidadora        198
babá              55
Name: count, dtype: int64


## 📝 Notas Finais

### ✅ Sites Implementados:
1. Catho
2. InfoJobs
3. Vagas.com
4. Trabalha Brasil
5. Indeed
6. LinkedIn (limitado)
7. Programathor
8. OLX
9. RemoteOK
10. Gupy
11. Jooble
12. Trampos.co

### ⚠️ Limitações:
- Sites com proteção anti-bot podem bloquear algumas requisições
- LinkedIn requer autenticação para resultados completos
- Alguns sites mudam estrutura HTML frequentemente
- Rate limiting pode afetar coleta em massa

### 🎯 Para o Projeto Íris:
- Este scraper coleta dados de **MUITO MAIS FONTES** que o código original
- Ideal para análise de padrões de aliciamento
- Dataset robusto para treinamento de modelos de ML
- Possibilita análise comparativa entre diferentes plataformas

### 🔄 Próximos Passos:
1. Implementar scraping incremental (atualização periódica)
2. Adicionar análise de sentimento nos textos
3. Criar dashboard de visualização
4. Implementar alertas para vagas suspeitas
5. Integrar com modelo de ML do Projeto Íris